# Lab Exercise: Encoding Categorical Data
This notebook explores the performance trade-offs, memory costs, and security risks (leakage) associated with different categorical encoding schemes under various ML models.

### Learning Objectives
1. **Performance Cost**: Measure how false numerical ordering affects linear versus tree models.
2. **Memory Footprint**: Quantify sparse vs. dense one-hot representation costs under high cardinality.
3. **Robustness**: Build encoding configurations robust to unseen categories at predict time.
4. **Safety**: Detect and fix target leakage when using category target encoding.

## Setup & Environment Initialization

In [ ]:
import warnings
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import (
    LabelEncoder,
    OneHotEncoder,
    OrdinalEncoder,
    TargetEncoder
)
from sklearn.compose import ColumnTransformer, make_column_selector

warnings.filterwarnings("ignore")
rng = np.random.RandomState(0)
print("✓ Environment initialized successfully.")

## 1. Does Label Encoding Hurt? (The Alternating Sign Problem)
Let's see how much label encoding nominal data costs when the category effects alternate in alphabetical order (the order encoders default to). We fit three models: a linear model (Logistic Regression), a shallow tree ensemble (max_depth=1), and an unrestricted tree ensemble.

In [ ]:
# Chennai, Delhi, Jaipur, Kolkata, Mumbai, Pune
effect = {"Chennai": 2.5, "Delhi": -2.0, "Jaipur": 2.4,
          "Kolkata": -1.8, "Mumbai": 2.2, "Pune": -2.1}
cities = list(effect)
city = rng.choice(cities, 3000)
y = (np.array([effect[c] for c in city]) + rng.normal(0, 1, 3000) > 0).astype(int)
df = pd.DataFrame({"city": city})

print("1. LABEL vs ONE-HOT — effect of each city, in alphabetical order:")
for i, c in enumerate(sorted(cities)):
    print(f"     {i} = {c:<9}{effect[c]:>6.1f}")
print("     -> the sign flips at every step: no single cut separates them\n")

ord_X = OrdinalEncoder().fit_transform(df[["city"]])
ohe_X = OneHotEncoder(sparse_output=False).fit_transform(df[["city"]])

print(f"   {'model':<26}{'label/ordinal':>15}{'one-hot':>10}{'gain':>9}")
for name, mk in [
        ("LogisticRegression", lambda: LogisticRegression(max_iter=2000)),
        ("Forest(max_depth=1)", lambda: RandomForestClassifier(max_depth=1, n_estimators=50, random_state=0)),
        ("Forest(unrestricted)", lambda: RandomForestClassifier(random_state=0))]:
    a = cross_val_score(mk(), ord_X, y, cv=5).mean()
    b = cross_val_score(mk(), ohe_X, y, cv=5).mean()
    print(f"   {name:<26}{a:>15.4f}{b:>10.4f}{b - a:>+9.4f}")
print(f"   majority-class baseline: {max(y.mean(), 1 - y.mean()):.4f}")

## 2. What Cardinality Costs
When cardinality (the number of unique categories) is high, one-hot encoding can result in a massive increase in columns. Let's compare dense versus sparse representations of high-cardinality columns.

In [ ]:
n = 50_000
print(f"2. COST OF CARDINALITY — one column, {n:,} rows")
print(f"   {'categories':>11}{'one-hot cols':>14}{'dense MB':>11}{'sparse MB':>11}{'binary cols':>13}")
for k in [3, 10, 50, 256, 1000]:
    col = rng.randint(0, k, n).astype(str).reshape(-1, 1)
    dense = OneHotEncoder(sparse_output=False).fit_transform(col)
    sp = OneHotEncoder(sparse_output=True).fit_transform(col)
    sp_mb = (sp.data.nbytes + sp.indices.nbytes + sp.indptr.nbytes) / 1e6
    print(f"   {k:>11}{dense.shape[1]:>14}{dense.nbytes / 1e6:>11.1f}"
          f"{sp_mb:>11.1f}{int(np.ceil(np.log2(k))):>13}")
print("\nbinary cols = ceil(log2(categories)); frequency encoding = always 1")

## 3. Unseen Categories at Predict Time
What happens when training on certain categories but a new category arrives at inference? Let's check which configurations raise exceptions and which recover gracefully.

In [ ]:
tr = pd.DataFrame({"city": ["Chennai", "Mumbai", "Delhi"] * 4})
te = pd.DataFrame({"city": ["Chennai", "Kolkata"]})      # 'Kolkata' is unseen

print("3. UNSEEN CATEGORY AT PREDICT TIME")
for label, enc in [
        ("LabelEncoder", LabelEncoder()),
        ("OrdinalEncoder (default)", OrdinalEncoder()),
        ("OrdinalEncoder(handle_unknown='use_encoded_value')",
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ("OneHotEncoder (default)", OneHotEncoder(sparse_output=False)),
        ("OneHotEncoder(handle_unknown='ignore')",
            OneHotEncoder(sparse_output=False, handle_unknown="ignore"))]:
    try:
        if isinstance(enc, LabelEncoder):
            out = enc.fit(tr["city"]).transform(te["city"])
        else:
            out = enc.fit(tr[["city"]]).transform(te[["city"]])
        print(f"   OK      {label}\n              -> {np.asarray(out).tolist()}")
    except Exception as e:
        print(f"   CRASH   {label}\n              -> {type(e).__name__}: "
              f"{str(e).splitlines()[0][:60]}")

## 4. Target Encoding & Leakage Prevention
Naive target encoding leaks information because the mean target is calculated over the full dataset, meaning each row's encoding includes its own label `y`. Let's prove how naive target encoding inflates accuracy on pure noise, and how scikit-learn's cross-fitted `TargetEncoder` prevents this leak.

In [ ]:
n2, k2 = 2000, 500
cat = rng.randint(0, k2, n2).astype(str).reshape(-1, 1)
y2 = rng.randint(0, 2, n2)

# Naive leakage calculation (over whole dataset)
means = pd.DataFrame({"c": cat.ravel(), "y": y2}).groupby("c")["y"].mean()
leaked = pd.Series(cat.ravel()).map(means).values.reshape(-1, 1)
naive = cross_val_score(LogisticRegression(), leaked, y2, cv=5).mean()

# Proper cross-fitted pipeline
proper = cross_val_score(
    make_pipeline(TargetEncoder(target_type="binary"), LogisticRegression()),
    cat, y2, cv=5).mean()

print("4. TARGET ENCODING ON A PURE-NOISE FEATURE (perfect score should be ~0.50)")
print(f"   {n2} rows, {k2} categories (~{n2 // k2} rows each), target is RANDOM")
print(f"   {'naive — category means from all the data':<48}{naive:>9.4f}   <- LEAKED")
print(f"   {'TargetEncoder in a Pipeline (cross-fitted)':<48}{proper:>9.4f}")

## Hands-On Programming Exercises

Complete the following coding challenges to practice robust encoding in production pipelines.

### Exercise 1: Build a Heterogeneous ColumnTransformer
Implement a `ColumnTransformer` that selectively targets columns:
- For all `object` (categorical) columns: Apply `OneHotEncoder(handle_unknown='ignore')`.
- For all numerical columns: Pass them through unchanged (use `'passthrough'`).

*Hint: Use `make_column_selector` to select columns by data type automatically.*

In [ ]:
df_ex = pd.DataFrame({
    'age': [25, 47, 31, 22],
    'income': [50000, 120000, 85000, 32000],
    'city': ['Chennai', 'Mumbai', 'Delhi', 'Chennai'],
    'vip': ['No', 'Yes', 'No', 'No']
})

# TODO: Construct and fit your ColumnTransformer below
ct = None

# out = ct.fit_transform(df_ex)
# print(out)

### Exercise 2: Ordered Ordinal Encoding
Encode a shirt size feature `['S', 'M', 'L', 'XL']` using `OrdinalEncoder` so that the numerical values correctly preserve the sizing order: `S=0, M=1, L=2, XL=3`.

*Hint: State your custom hierarchy explicitly via the `categories` parameter.*

In [ ]:
sizes = pd.DataFrame({'size': ['L', 'S', 'XL', 'M', 'S', 'L']})

# TODO: Fit OrdinalEncoder with explicit ordering and transform
encoder = None

# encoded = encoder.fit_transform(sizes)
# print(encoded)

### Exercise 3: Memory Profiler for One-Hot Encoding
Construct a loop over category counts `k = [10, 100, 500, 1000]`. Generate a mock categorical series of 100,000 rows with `k` unique values. Compare the memory usage of the resulting dense numpy array versus the scipy sparse matrix representation.

In [ ]:
n_rows = 100_000
for k in [10, 100, 500, 1000]:
    # TODO: Create mock column, run OneHotEncoder dense vs sparse, and print memory usage in MB.
    pass